In [2]:
from google.colab import files
files.upload()

Saving BCICIV_2b_gdf.zip to BCICIV_2b_gdf.zip


In [3]:
!unzip BCICIV_2b_gdf.zip -d /content/data

Archive:  BCICIV_2b_gdf.zip
  inflating: /content/data/B0101T.gdf  
  inflating: /content/data/B0102T.gdf  
  inflating: /content/data/B0103T.gdf  
  inflating: /content/data/B0104E.gdf  
  inflating: /content/data/B0105E.gdf  
  inflating: /content/data/B0201T.gdf  
  inflating: /content/data/B0202T.gdf  
  inflating: /content/data/B0203T.gdf  
  inflating: /content/data/B0204E.gdf  
  inflating: /content/data/B0205E.gdf  
  inflating: /content/data/B0301T.gdf  
  inflating: /content/data/B0302T.gdf  
  inflating: /content/data/B0303T.gdf  
  inflating: /content/data/B0304E.gdf  
  inflating: /content/data/B0305E.gdf  
  inflating: /content/data/B0401T.gdf  
  inflating: /content/data/B0402T.gdf  
  inflating: /content/data/B0403T.gdf  
  inflating: /content/data/B0404E.gdf  
  inflating: /content/data/B0405E.gdf  
  inflating: /content/data/B0501T.gdf  
  inflating: /content/data/B0502T.gdf  
  inflating: /content/data/B0503T.gdf  
  inflating: /content/data/B0504E.gdf  
  inflating:

In [4]:
!pip install tensorflow

In [6]:
import numpy as np

# Placeholder for X_all and y_all based on inferred shapes from later cells.
# In a real scenario, you would load your EEG data from the .gdf files here,
# for example, using the MNE library to read GDF files and extract epochs/trials.
# Example inferred shape for X_all: (trials, channels, samples) -> (400, 6, 501)
X_all = np.random.rand(400, 6, 501)  # Dummy data for demonstration
y_all = np.random.randint(0, 2, 400)  # Dummy labels (0 or 1 for 2 classes)


X_dl = X_all.copy()

# normalize (VERY IMPORTANT)
X_dl = (X_dl - X_dl.mean(axis=2, keepdims=True)) / (
        X_dl.std(axis=2, keepdims=True) + 1e-6)

# add channel dimension
X_dl = X_dl[..., np.newaxis]

print("EEGNet input shape:", X_dl.shape)

EEGNet input shape: (400, 6, 501, 1)


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_eegnet(n_channels, n_samples, n_classes=2):

    input1 = layers.Input(shape=(n_channels, n_samples, 1))

    # Temporal convolution
    x = layers.Conv2D(8, (1, 64), padding='same', use_bias=False)(input1)
    x = layers.BatchNormalization()(x)

    # Depthwise spatial convolution (like CSP)
    x = layers.DepthwiseConv2D((n_channels, 1),
                               depth_multiplier=2,
                               use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 4))(x)
    x = layers.Dropout(0.5)(x)

    # Separable convolution
    x = layers.SeparableConv2D(16, (1, 16),
                               padding='same',
                               use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 8))(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Flatten()(x)
    output = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs=input1, outputs=output)
    return model

In [8]:
X_eegnet = X_all[..., np.newaxis]

print("EEGNet input shape:", X_eegnet.shape)

EEGNet input shape: (400, 6, 501, 1)


In [9]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.layers import AveragePooling2D, Dropout
from tensorflow.keras.layers import SeparableConv2D, Flatten, Dense
from tensorflow.keras.constraints import max_norm

def build_eegnet(nb_classes=2, Chans=22, Samples=500):

    input1 = Input(shape=(Chans, Samples, 1))

    # Block 1
    x = Conv2D(16, (1, 64), padding='same', use_bias=False)(input1)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D(
        (Chans, 1),
        use_bias=False,
        depth_multiplier=2,
        depthwise_constraint=max_norm(1.)
    )(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(0.5)(x)

    # Block 2
    x = SeparableConv2D(16, (1, 16), use_bias=False, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(0.5)(x)

    x = Flatten()(x)
    output = Dense(nb_classes, activation='softmax')(x)

    model = Model(inputs=input1, outputs=output)
    return model

In [10]:
model = build_eegnet(
    nb_classes=2,
    Chans=X_all.shape[1],
    Samples=X_all.shape[2]
)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6, 501, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 6, 501, 16)     │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 6, 501, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 1, 501, 32)     │           192 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 501, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1, 501, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 1, 125, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 125, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 1, 125, 16)     │         1,024 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 125, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 1, 125, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 1, 15, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 15, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 240)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           482 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,978 (11.63 KB)

 Trainable params: 2,850 (11.13 KB)

 Non-trainable params: 128 (512.00 B)

In [11]:
mean = X_all.mean()
std = X_all.std()

X_all = (X_all - mean) / std
X_eegnet = X_all[..., np.newaxis]

In [12]:
history = model.fit(
    X_eegnet,
    y_all,
    batch_size=16,
    epochs=50,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - accuracy: 0.5097 - loss: 0.7748 - val_accuracy: 0.4625 - val_loss: 0.6964
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.5755 - loss: 0.7171 - val_accuracy: 0.4625 - val_loss: 0.6970
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.5796 - loss: 0.6938 - val_accuracy: 0.4625 - val_loss: 0.6991
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - accuracy: 0.5843 - loss: 0.6995 - val_accuracy: 0.4625 - val_loss: 0.6989
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 119ms/step - accuracy: 0.5331 - loss: 0.7093 - val_accuracy: 0.4625 - val_loss: 0.6989
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.6259 - loss: 0.6437 - val_accuracy: 0.4625 - val_loss: 0.6991
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - accuracy: 0.6575 - loss: 0.6399 - val_accuracy: 0.4750 - val_loss: 0.7001
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.5500 - loss: 0.6911 - val_accuracy: 0.4875 - 

In [13]:
loss, acc = model.evaluate(X_eegnet, y_all, verbose=0)
print("EEGNet Accuracy: %.2f%%" % (acc*100))

EEGNet Accuracy: 88.25%


In [17]:
trial_id = 15

pred_prob = model.predict(X_eegnet[trial_id:trial_id+1])
pred_label = np.argmax(pred_prob)

print("True label:", y_all[trial_id])
print("Predicted label:", pred_label)
print("Probabilities:", pred_prob)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
True label: 0
Predicted label: 0
Probabilities: [[0.6023312  0.39766878]]
